## 1. 内置中间件ModelFallbackMiddleware演示

In [1]:
import  os
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

### 1.1 定义主备模型客户端

In [2]:
deepseek_primary = ChatOpenAI(
    model="deepseek-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    max_retries=0,
    timeout=15
)

qwen_backup = ChatOpenAI(
    model="qwen3.8-max-0902",
    api_key=os.getenv("QW_API_KEY"),
    base_url=os.getenv("QW_BASE_URL"),
    max_retries=0,
    timeout=15
)

### 1.2 主模型故障

In [3]:
# 主模型故障（错误 Key） -> 自动降级到 Qwen
deepseek_broken = ChatOpenAI(
    model="deepseek-flash",
    api_key="sk-invalid-key",
    base_url="https://api.deepseek.com",
    timeout=15
)


### 1.3 定义降级中间件

In [4]:
# middleware 里放备用模型
fallback_mw = ModelFallbackMiddleware(qwen_backup)


### 1.4 测试

In [5]:
for label, primary in [("正常", deepseek_primary), ("降级", deepseek_broken)]:
    agent = create_agent(model=primary, tools=[], middleware=[fallback_mw])
    res = agent.invoke({"messages": [HumanMessage(content="你好，请回复收到")]})
    msg = res["messages"][-1]
    print(f"[{label}] 实际响应模型: {msg.response_metadata.get('model_name')} | 回复: {msg.content}")


[正常] 实际响应模型: deepseek-flash | 回复: 收到
[降级] 实际响应模型: qwen3.8-max-0902 | 回复: 收到
